# Does OpenAI's Batch API execute the hosted `web_search` tool?

Feasibility probe for adding `judge_backend: web_search` support to the batch path.

The live judge sends a `/v1/responses` request with `tools=[{"type": "web_search", ...}]`.
The batch path today only emits `/v1/chat/completions` lines with no tools. Before building
a parallel responses-batch lane through `batch_api.py` + `retrieve_batch_results.py`, this
notebook answers the one unknown: **will OpenAI actually run web_search inside a batch job?**

**Steps**
1. Control: a *live* responses call with web_search → confirms search works for this key/model.
2. Probe: a one-line *batch* against `/v1/responses` with web_search → poll → inspect output.

**Reading the verdict**
- Batch creation rejected (`BadRequestError` on `endpoint`/`tools`) → **not supported**.
- Batch completes but output has **no** `web_search_call` item → tool silently ignored → **effectively not supported** (same failure mode as today).
- Batch completes **with** a `web_search_call` item → **supported**; the responses-batch lane is worth building.

In [ ]:
import os, json, time, toml
from openai import OpenAI

# Locate secrets.toml exactly like the repo (SECRETS env var, else walk up to repo root).
secrets_path = os.getenv("SECRETS")
if not secrets_path:
    here = os.getcwd()
    for _ in range(6):
        cand = os.path.join(here, "secrets.toml")
        if os.path.exists(cand):
            secrets_path = cand
            break
        here = os.path.dirname(here)
assert secrets_path and os.path.exists(secrets_path), (
    f"secrets.toml not found — set the SECRETS env var. Got: {secrets_path!r}"
)
secrets = toml.load(secrets_path)
client = OpenAI(api_key=secrets["openai_key"])
print("OpenAI client ready. secrets:", secrets_path)

# --- knobs ---
MODEL = "gpt-5.5-pro"            # change to "gpt-5.5-pro" to match the ablation exactly (slower/pricier)
WEB_SEARCH_TOOL = {"type": "web_search", "filters": {"allowed_domains": ["arxiv.org"]}}
# A prompt that can only be answered by actually searching (forces a tool call if available).
PROMPT = (
    "Use web search to find a recent arXiv paper on retrieval-augmented LLM evaluation. "
    "Return ONLY its exact title and arXiv id. You must search; do not answer from memory."
)

def output_item_types(items):
    """List the .type of each Responses output item (SDK objects or batch JSON dicts)."""
    types = []
    for it in items:
        t = getattr(it, "type", None) if not isinstance(it, dict) else it.get("type")
        types.append(t)
    return types

## Step 1 — Control: live call (proves web_search works at all)

If this shows `web_search_call` but the batch later does not, the gap is batch-specific.

In [ ]:
resp = client.responses.create(
    model=MODEL,
    input=PROMPT,
    tools=[WEB_SEARCH_TOOL],
    reasoning={"effort": "medium"},
)
types = output_item_types(resp.output)
print("output item types:", types)
print("web_search executed (LIVE)?", "web_search_call" in types)
print("\n--- answer ---\n", (resp.output_text or "")[:500])

## Step 2 — Probe: submit a one-line batch to `/v1/responses`

A `BadRequestError` here (on the endpoint or the tool) is already a conclusive **"not supported"**.

In [ ]:
req = {
    "custom_id": "websearch-probe-1",
    "method": "POST",
    "url": "/v1/responses",
    "body": {
        "model": MODEL,
        "input": PROMPT,
        "tools": [WEB_SEARCH_TOOL],
        "reasoning": {"effort": "medium"},
    },
}
fpath = "web_search_batch_probe_requests.jsonl"
with open(fpath, "w") as f:
    f.write(json.dumps(req) + "\n")

batch_id = None
try:
    up = client.files.create(file=open(fpath, "rb"), purpose="batch")
    batch = client.batches.create(
        input_file_id=up.id,
        endpoint="/v1/responses",
        completion_window="24h",
    )
    batch_id = batch.id
    print("submitted. batch id:", batch_id, "| status:", batch.status)
except Exception as e:
    print("VERDICT: batch creation REJECTED -> web_search not supported in batch via this path.")
    print(type(e).__name__, "-", e)

## Step 3 — Poll until terminal

Small batches usually finish in a few minutes. Re-run this cell if it's still `in_progress`.

In [ ]:
assert batch_id, "No batch_id — Step 2 failed (see its output)."
TERMINAL = {"completed", "failed", "expired", "cancelled"}
MAX_WAIT_S = 20 * 60
start = time.time()
while True:
    b = client.batches.retrieve(batch_id)
    print(time.strftime("%H:%M:%S"), b.status, "| counts:", b.request_counts)
    if b.status in TERMINAL or (time.time() - start) > MAX_WAIT_S:
        break
    time.sleep(15)
print("\nstopped at status:", b.status)

## Step 4 — Inspect output and print the verdict

In [ ]:
b = client.batches.retrieve(batch_id)
if b.status != "completed":
    print("Batch did not complete:", b.status)
    if getattr(b, "errors", None):
        print(b.errors)
    if getattr(b, "error_file_id", None):
        print("--- error file ---")
        print(client.files.content(b.error_file_id).text[:2000])
else:
    out = client.files.content(b.output_file_id).text
    line = json.loads(out.splitlines()[0])
    body = line["response"]["body"]
    types = output_item_types(body.get("output", []))
    did_search = "web_search_call" in types
    print("output item types:", types)
    print("usage:", body.get("usage"))
    print("\n--- answer ---\n", (body.get("output_text") or "")[:500])
    print("\n" + "=" * 60)
    print("VERDICT: web_search executed in BATCH?", did_search)
    print("=" * 60)
    if did_search:
        print("-> Supported. Building the responses-batch lane (Option A) is worthwhile.")
    else:
        print("-> Completed but never searched: tool silently ignored. Treat as NOT supported.")